# Experiment 5: Red Wine Quality Linear Model

**Objective**: Define and train a linear model for the Red Wine Quality dataset.

**Dataset**: Red Wine Quality Dataset (Kaggle/UCI)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import subprocess
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Load environment variables
load_dotenv()

print("Libraries imported successfully.")

## 2. Download Dataset from Kaggle

In [ ]:
# Download Red Wine Quality dataset from Kaggle
dataset_path = '../data/winequality-red.csv'

if not os.path.exists(dataset_path):
    print("Downloading Red Wine Quality dataset from Kaggle...")
    try:
        subprocess.run([
            'kaggle', 'datasets', 'download', '-d', 
            'uciml/red-wine-quality-cortez-et-al-2009',
            '-p', '../data', '--unzip'
        ], check=True, capture_output=True)
        print("Dataset downloaded successfully!")
    except Exception as e:
        print(f"Error downloading: {e}")
        print("Attempting UCI direct download...")
        try:
            import urllib.request
            url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
            urllib.request.urlretrieve(url, dataset_path)
            print("Downloaded from UCI repository.")
        except Exception as e2:
            print(f"Direct download also failed: {e2}")
            print("Creating synthetic wine dataset...")
            n_samples = 1599
            np.random.seed(42)
            data = {
                'fixed acidity': np.random.uniform(4.6, 15.9, n_samples),
                'volatile acidity': np.random.uniform(0.12, 1.58, n_samples),
                'citric acid': np.random.uniform(0, 1, n_samples),
                'residual sugar': np.random.uniform(0.9, 15.5, n_samples),
                'chlorides': np.random.uniform(0.012, 0.611, n_samples),
                'free sulfur dioxide': np.random.uniform(1, 72, n_samples),
                'total sulfur dioxide': np.random.uniform(6, 289, n_samples),
                'density': np.random.uniform(0.990, 1.004, n_samples),
                'pH': np.random.uniform(2.74, 4.01, n_samples),
                'sulphates': np.random.uniform(0.33, 2, n_samples),
                'alcohol': np.random.uniform(8.4, 14.9, n_samples)
            }
            df = pd.DataFrame(data)
            df['quality'] = np.clip(
                5 + 0.3 * df['alcohol'] - 1.5 * df['volatile acidity'] + 
                0.2 * df['citric acid'] + np.random.normal(0, 0.5, n_samples),
                3, 8
            ).astype(int)
            df.to_csv(dataset_path, index=False, sep=';')
else:
    print("Dataset already exists.")

## 3. Load and Explore Dataset

In [ ]:
# Try loading with different separators
try:
    df = pd.read_csv(dataset_path, sep=';')
    if len(df.columns) == 1:  # CSV stored with comma separator
        df = pd.read_csv(dataset_path)
except:
    df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Dataset info
print("\nDataset Info:")
df.info()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Quality distribution
print("\nQuality Distribution:")
print(df['quality'].value_counts().sort_index())

## 4. Data Preprocessing

In [ ]:
# Handle missing values if any
df = df.dropna()
print(f"Dataset shape after cleaning: {df.shape}")

In [ ]:
# Separate features and target
X = df.drop('quality', axis=1)
y = df['quality']

feature_names = X.columns.tolist()

print(f"Features: {feature_names}")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied successfully.")
print(f"\nScaled Features Mean (train): {X_train_scaled.mean(axis=0).round(6)}")
print(f"Scaled Features Std (train): {X_train_scaled.std(axis=0).round(4)}")

## 5. Data Visualization

In [ ]:
# Quality distribution and correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Quality distribution
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], 
                                                color='darkred', edgecolor='black')
axes[0].set_title('Wine Quality Distribution', fontweight='bold')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Feature correlation with quality
corr_with_quality = df.corr()['quality'].drop('quality').sort_values()
colors = ['darkred' if x < 0 else 'darkgreen' for x in corr_with_quality]
corr_with_quality.plot(kind='barh', ax=axes[1], color=colors)
axes[1].set_title('Feature Correlation with Quality', fontweight='bold')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('../data/wine_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, 
            fmt='.2f', square=True, mask=mask, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/wine_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Define and Train Linear Model

In [ ]:
# Simple Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

print("Linear Regression Model Trained!")
print(f"\nIntercept: {linear_model.intercept_:.4f}")
print("\nCoefficients:")
for name, coef in zip(feature_names, linear_model.coef_):
    print(f"  {name}: {coef:.4f}")

In [ ]:
# Cross-validation score
cv_scores = cross_val_score(linear_model, X_train_scaled, y_train, cv=5, scoring='r2')
print(f"\nCross-Validation R² Scores: {cv_scores.round(4)}")
print(f"Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## 7. Model Evaluation

In [ ]:
# Predictions
y_pred_train = linear_model.predict(X_train_scaled)
y_pred_test = linear_model.predict(X_test_scaled)

# Evaluation metrics
metrics = {
    'train': {
        'mse': mean_squared_error(y_train, y_pred_train),
        'rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'mae': mean_absolute_error(y_train, y_pred_train),
        'r2': r2_score(y_train, y_pred_train)
    },
    'test': {
        'mse': mean_squared_error(y_test, y_pred_test),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'mae': mean_absolute_error(y_test, y_pred_test),
        'r2': r2_score(y_test, y_pred_test)
    }
}

print("Linear Model Evaluation:")
print("="*50)
for split in ['train', 'test']:
    print(f"\n{split.upper()} Set:")
    print(f"  MSE: {metrics[split]['mse']:.4f}")
    print(f"  RMSE: {metrics[split]['rmse']:.4f}")
    print(f"  MAE: {metrics[split]['mae']:.4f}")
    print(f"  R²: {metrics[split]['r2']:.4f}")

## 8. Visualization of Results

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_pred_train, alpha=0.5, edgecolors='k', linewidth=0.3, color='darkred')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'b--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Quality')
axes[0].set_ylabel('Predicted Quality')
axes[0].set_title(f'Training Set (R² = {metrics["train"]["r2"]:.4f})', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_pred_test, alpha=0.5, edgecolors='k', linewidth=0.3, color='darkred')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'b--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Quality')
axes[1].set_ylabel('Predicted Quality')
axes[1].set_title(f'Test Set (R² = {metrics["test"]["r2"]:.4f})', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/wine_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (absolute coefficients)
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': linear_model.coef_,
    'Abs_Coefficient': np.abs(linear_model.coef_)
}).sort_values('Abs_Coefficient', ascending=True)

plt.figure(figsize=(10, 6))
colors = ['darkred' if x < 0 else 'darkgreen' for x in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.xlabel('Coefficient Value')
plt.title('Linear Model Feature Coefficients', fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../data/wine_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual analysis
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_test, residuals, alpha=0.5, edgecolors='k', linewidth=0.3, color='darkred')
axes[0].axhline(y=0, color='blue', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Quality')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='darkred')
axes[1].axvline(x=0, color='blue', linestyle='--', linewidth=2)
axes[1].axvline(x=residuals.mean(), color='green', linestyle='-.', linewidth=2, 
                label=f'Mean: {residuals.mean():.3f}')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/wine_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 5 SUMMARY: Red Wine Quality Linear Model")
print("="*60)

print("\nDataset Information:")
print(f"- Total samples: {len(df)}")
print(f"- Features: {len(feature_names)}")
print(f"- Target: Quality score (3-8)")

print("\nLinear Model Performance:")
print(f"- Training R²: {metrics['train']['r2']:.4f}")
print(f"- Test R²: {metrics['test']['r2']:.4f}")
print(f"- Test RMSE: {metrics['test']['rmse']:.4f}")
print(f"- Test MAE: {metrics['test']['mae']:.4f}")
print(f"- Cross-Validation R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Top features
top_features = coef_df.nlargest(3, 'Abs_Coefficient')[['Feature', 'Coefficient']]
print("\nMost Important Features:")
for _, row in top_features.iterrows():
    direction = "positive" if row['Coefficient'] > 0 else "negative"
    print(f"- {row['Feature']}: {row['Coefficient']:.4f} ({direction} impact)")

print("\nKey Observations:")
print("- Linear regression provides interpretable coefficients")
print("- Alcohol content has strong positive correlation with quality")
print("- Volatile acidity has negative impact on quality")
print("- Feature scaling ensures fair comparison of coefficients")